## Libraries

In [ ]:
import pandas as pd
from sklearn import linear_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
import matplotlib.colors
import matplotlib.cm
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

pd.set_option('display.float_format', lambda x: '%.2f' % x) #virgülden sonra 2
pd.options.display.max_columns = None #tüm kolonlar gözüksün

## About Data

### Columns

#### df_header

- **individualnumber** : Unique customer number
- **wholesale_store_shopping_count** : The number of shopping from wholesale store. Integer.
- **item_count** : Total number of items purchased. Integer.
- **distinct_item_count** : Total number of distinct items purchased. Integer.
- **quantity_unit_more_than_12** : The number of purchased more than 12 from an item. Integer.
- **online_shopping** : Total amount of online shopping. Float.
- **offline_shopping** : Total amount of offline shopping. Float.
- **total_sales** : Total amount of shopping. Float.
- **is_wholesaler** : Customer is wholesaler or not. Integer.


#### df_discount

- **individualnumber** : Unique customer number
- **total_discount_item_count** : Total number of items purchased that have discount. Integer.
- **online_discount** : Total amount of online shopping with discount. Float.
- **offline_discount** : Total amount of offline shopping with discount. Float.
- **total_discount** : Total amount of shopping with discount. Float.

#### df_segment

- **individualnumber** : Unique customer number
- **segment** : Customer segmentation. Integer.

#### daily_sales

- **date_of_transaction** : Transaction date like 2022-05-22. String.
- **total_sales** : Total amount of shopping. Float.
- **total_items** : Total number of items that are purchased. Integer.

In [ ]:
# Controlling the data types
print(df_header.dtype)
print(df_discount.dtype)
print(df_segment.dtype)
print(daily_sales.dtype)

In [ ]:
# About the data
print(df_header.info())
print(df_discount.info())
print(df_segment.info())
print(daily_sales.info())

## Data Preparation

In [ ]:
# Control the Nan values
df_header.isnull().sum()

In [ ]:
# Drop the Nan values
df_header = df_header.dropna()

In [ ]:
df_header.describe()

In [ ]:
# Merging the dataframes
df_wholesale = pd.merge(df_header, df_discount, how="left", on="individualnumber")
df_wholesale = pd.merge(df_wholesale, df_segment, how="left", on="individualnumber")

In [ ]:
# Looking summary statistics of the merged dataframe
df_wholesale.describe()

In [ ]:
df_to_be_used = df_wholesale.copy()

In [ ]:
# As Target, I flag those who have shopped at least 20 times from wholesale stores 
# or whose is_wholesaler column is full as 1, and the rest as 0.
# I also flag those who appear as wholesale in the segment as 1.
df_wholesale["target"] =0
df_wholesale.loc[(np.logical_or(df_wholesale["wholesale_store_shopping_count"]>=18, df_wholesale["is_wholesaler"]==1)),"target"] =1
df_wholesale.loc[df_wholesale["segment"]==1, "target"] =1

In [ ]:
# Looking for the target variable distribution
df_wholesale["target"].value_counts()

In [ ]:
# Calculating the median of the numeric variable which is target variable equal to 1
df_wholesale[df_wholesale["target"]==1].median()

In [ ]:
# Drop the columns that are using for the target variable
df_to_be_dropped = df_wholesale.copy()
df_wholesale_dropped = df_to_be_dropped.drop(["is_wholesaler"], axis=1)
df_wholesale_dropped = df_wholesale_dropped.drop(["deger_segment"], axis=1)

In [ ]:
# Controlling Nan values again
df_wholesale_dropped.isna().any()

In [ ]:
# Filling the NaN values with 0 because if i fill with mean or median, it will affect the model
df_wholesale = df_wholesale_dropped.fillna(0)

In [ ]:
# Filter the dataframe for customers who have purchased at least 1 item
df_wholesale_dropped = df_wholesale_dropped[df_wholesale_dropped["item_count"]>0] 

In [ ]:
# Changing the data types of the columns
# Convert columns to int
df_wholesale_dropped["individualnumber"] = df_wholesale_dropped.individualnumber.astype("int")
# Convert columns to float
cols_to_float = ["online_satis", "offline_satis", "total_sales", "online_discount", "offline_discount", "total_discount"]
for col in cols_to_float:
    df_wholesale_dropped[col] = df_wholesale_dropped[col].astype("float")

In [ ]:
# Display the first few rows of the dataframe
df_wholesale_dropped.head()

## Günlük Anomaly

In [ ]:
#Customers who can be considered as wholesalers usually prefer days with promotions.
#With Isolation Forest I add a new column to the table by detecting days with abnormal sales during the year.

In [ ]:
# Convert total_sales to float
daily_sales["total_sales"] = daily_sales.total_sales.astype("int")

# Sorting the daily_sales dataframe by date_of_transaction
daily_sales = daily_sales.sort_values("date_of_transaction")

# Plotting the daily sales data
plt.plot(daily_sales["date_of_transaction"], daily_sales["total_sales"])

In [ ]:
#The data that i use is 2023-01-01 to 2024-12-31
# I separate the two years to avoid any anomalies arising from price differences in the two years.
df_2023 = daily_sales[daily_sales["date_of_transaction"]<"2024-01-01"]
df_2024 = daily_sales[daily_sales["date_of_transaction"]>="2024-01-01"]

In [ ]:
# Display the first few rows of the dataframe
df_2023.head()

In [ ]:
#The Isolation Forest algorithm determines an anomaly score for all days in the year
# by looking at the total daily sales and the total number of products sold.
#With this score, it assigns -1 to abnormal days and 1 to normal days.
from sklearn.ensemble import IsolationForest

inputs = ['total_sales', 'total_items']
iso_forest = IsolationForest(contamination = 0.04, random_state=42)
iso_forest.fit(df_2023[inputs])
df_2023['anomaly_score'] = iso_forest.decision_function(df_2023[inputs])
df_2023['anomaly'] = iso_forest.predict(df_2023[inputs])

In [ ]:
#Looking the anomaly days distribution
df_2023["anomaly"].value_counts()

In [ ]:
# Filter the dataframe for anomaly days
anomaly_sales_2023 = df_2023[df_2023["anomaly"] == -1]
anomaly_sales_2023.sort_values("date_of_transaction")

In [ ]:
# Listing the dates of the anomaly days
list_2023 = anomaly_sales_2023["date_of_transaction"].tolist()

In [ ]:
#Same thing for 2024
#The Isolation Forest algorithm determines an anomaly score for all days in the year
from sklearn.ensemble import IsolationForest

inputs = ['total_sales', 'total_items']
iso_forest = IsolationForest(contamination = 0.04, random_state=42)
iso_forest.fit(df_2024[inputs])
df_2024['anomaly_score'] = iso_forest.decision_function(df_2024[inputs])
df_2024['anomaly'] = iso_forest.predict(df_2024[inputs])

In [ ]:
# Looking the anomaly days distribution
df_2024["anomaly"].value_counts()

In [ ]:
# Filter the dataframe for anomaly days
anomaly_sales_2024 = df_2024[df_2024["anomaly"] == -1]
anomaly_sales_2024.sort_values("date_of_transaction")

In [ ]:
# Listing the dates of the anomaly days
list_2024 = anomaly_sales_2024["date_of_transaction"].tolist()

In [ ]:
# Concatenating the two lists of dates
date_list = list_2023 + list_2024
date_list

In [ ]:
set_date = tuple(date_list)

In [ ]:
# Listing the customer numbers who is in df_wholesale_dropped
ind_number = df_wholesale_dropped["individualnumber"].tolist()

In [ ]:
set_ind = tuple(ind_number)

In this part I am taking sales made by customers on abnormal days

**sum_df**
- individual_number
- special_date_amount: Float


In [ ]:
# Merge the wholesale data with the anomaly data
df_new = pd.merge(df_wholesale_dropped, sum_df, on="individualnumber", how="left")

In [ ]:
# Filling the NaN values (the customer that are not purchasing anythin on anomaly days) with 0
df_new["special_date_amount"].fillna(0, inplace=True)

In [ ]:
# Display the first few rows of the dataframe
df_new.head()

In [ ]:
# Dropping the duplicate rows and NaN values
df_new = df_new.drop_duplicates().dropna()

In [ ]:
# Looking the summary statistics of the new dataframe
df_new.describe()

## KNN

In [ ]:
merged_df = df_new.copy()

In [ ]:
# Setting the X and y variables for the model
X = merged_df.drop(['target'],axis=1)
y = merged_df[['target']]

In [ ]:
# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
# Creates a list, positive_indices, containing the indices of rows in y["target"]
#  where the value of target is equal to 1.
positive_indices = [i for i, target in enumerate(y["target"]) if target == 1]

In [ ]:
# Selects the rows from X corresponding to the indices in positive_indices and stores them in positive_instances
positive_instances = X.iloc[positive_indices]

In [ ]:
# initializes a k-Nearest Neighbors (k-NN) classifier with n_neighbors=2,
#  trains it on the training data (X_train, y_train),
#  makes predictions on the test data (X_test), prints the accuracy score of the model, 
# and stores the predicted values in y_pred and the true labels in y_true.
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors = 2) 
knn.fit(X_train,y_train)
prediction = knn.predict(X_test)
print(" {} nn score: {} ".format(2,knn.score(X_test,y_test)))

y_pred = knn.predict(X_test)
y_true=y_test

In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix
cm= confusion_matrix(y_true, y_pred)

In [ ]:
# Plotting the confusion matrix

import seaborn as sns 
f, ax =plt.subplots(figsize = (5,5))

sns.heatmap(cm,annot = True, linewidths= 0.5, fmt=".0f", ax=ax)
plt.xlabel("y_pred")
plt.ylabel("y_true")
plt.show()

In [ ]:
#  NearestNeighbors model with n_neighbors=2 to find the 2 nearest neighbors for each data point
#  and fits it to the dataset
from sklearn.neighbors import NearestNeighbors
classifier = NearestNeighbors(n_neighbors = 2)
classifier.fit(X)

In [ ]:
# finds the indices of the 2 nearest neighbors for each data point in positive_instances without returning the distances
similar_indices = classifier.kneighbors(positive_instances, return_distance=False)

#finds both the indices and the distances of the 2 nearest neighbors for each data point in positive_instances.
similar_indices_with_distances = classifier.kneighbors(positive_instances, return_distance=True)

In [ ]:
# Flattens the array of indices to a 1D array
similar_instances = X.iloc[similar_indices.flatten()]

In [ ]:
# Filter the DataFrame to only include the necessary columns
df_filtered = df_to_be_used[["individualnumber", "is_wholesaler","deger_segment"]]

# Perform the merge operation
output_df = similar_instances.merge(df_filtered, on="individualnumber", how="left")